<a href="https://colab.research.google.com/github/Nethaji2606/thinkpalm-agentai-Nethaji-GetItOwn_2.0/blob/main/Download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Bike Comparison and EMI Tool — show specs immediately after brand+model selection
# Updated: brand and model dropdowns start with "Select option" placeholders (no defaults)
import pandas as pd
import math
import io
from IPython.display import display, HTML, FileLink
import ipywidgets as widgets

# Data model
BIKES = {
    "Royal Enfield": {
        "Classic 350": {
            "Ex-Showroom Price": 193000,
            "Engine Displacement cc": 349,
            "Max Power bhp": 20.2,
            "Max Torque Nm": 27,
            "Fuel Tank Capacity L": 13,
            "Mileage kmpl": 35
        },
        "Meteor 350": {
            "Ex-Showroom Price": 200000,
            "Engine Displacement cc": 349,
            "Max Power bhp": 20.2,
            "Max Torque Nm": 27,
            "Fuel Tank Capacity L": 15,
            "Mileage kmpl": 37
        }
    },
    "Yamaha": {
        "R15 V4": {
            "Ex-Showroom Price": 182000,
            "Engine Displacement cc": 155,
            "Max Power bhp": 18.4,
            "Max Torque Nm": 14.1,
            "Fuel Tank Capacity L": 11,
            "Mileage kmpl": 45
        },
        "FZ 25": {
            "Ex-Showroom Price": 160000,
            "Engine Displacement cc": 249,
            "Max Power bhp": 20.8,
            "Max Torque Nm": 20,
            "Fuel Tank Capacity L": 14,
            "Mileage kmpl": 40
        }
    },
    "KTM": {
        "Duke 390": {
            "Ex-Showroom Price": 311000,
            "Engine Displacement cc": 399,
            "Max Power bhp": 43.5,
            "Max Torque Nm": 37,
            "Fuel Tank Capacity L": 13.4,
            "Mileage kmpl": 28
        },
        "RC 200": {
            "Ex-Showroom Price": 250000,
            "Engine Displacement cc": 199,
            "Max Power bhp": 25,
            "Max Torque Nm": 19.2,
            "Fuel Tank Capacity L": 10,
            "Mileage kmpl": 35
        }
    }
}

# Utilities
def format_currency(x):
    try:
        return f"₹{int(round(float(x))):,}"
    except Exception:
        return x

def calculate_emi(principal, annual_rate_percent, months):
    r = annual_rate_percent / 12 / 100
    if months == 0:
        return 0.0
    if r == 0:
        return principal / months
    emi = principal * r * (1 + r) ** months / ((1 + r) ** months - 1)
    return emi

def build_comparison_df(selected_models):
    specs = [
        "Ex-Showroom Price",
        "Engine Displacement cc",
        "Max Power bhp",
        "Max Torque Nm",
        "Fuel Tank Capacity L",
        "Mileage kmpl"
    ]
    data = {}
    for brand, model in selected_models:
        key = f"{brand} {model}"
        specs_dict = BIKES[brand][model]
        col = [specs_dict.get(s, None) for s in specs]
        data[key] = col
    df = pd.DataFrame(data, index=specs)
    return df

def emi_breakdown_for_model(brand, model, down_payment=0.0, annual_rate=10.0, tenures=[3,6,9,12,36,60]):
    price = float(BIKES[brand][model]["Ex-Showroom Price"])
    loan_amount = max(0.0, price - down_payment)
    rows = []
    for months in tenures:
        emi = calculate_emi(loan_amount, annual_rate, months)
        total_payable = emi * months
        total_interest = total_payable - loan_amount
        rows.append({
            "Tenure Months": months,
            "Monthly EMI": format_currency(emi),
            "Total Interest": format_currency(total_interest),
            "Total Amount Payable": format_currency(total_payable),
            "Loan Amount": format_currency(loan_amount)
        })
    return pd.DataFrame(rows)

# Widgets with "Select option" placeholders (value=None)
brand_options = [("Select Brand", None)] + [(b, b) for b in sorted(BIKES.keys())]
brand_dropdown = widgets.Dropdown(options=brand_options, value=None, description="Brand")

model_options_initial = [("Select Model", None)]
model_dropdown = widgets.Dropdown(options=model_options_initial, value=None, description="Model")

compare_selector = widgets.SelectMultiple(options=[], description="Compare", rows=8)
down_payment_input = widgets.BoundedFloatText(value=0.0, min=0.0, max=1e9, description="Down Payment ₹")
interest_input = widgets.BoundedFloatText(value=10.0, min=0.0, max=100.0, step=0.1, description="Annual %")
compare_button = widgets.Button(description="Show Comparison", button_style="primary")
emi_button = widgets.Button(description="Show EMI", button_style="info")
export_compare_btn = widgets.Button(description="Export Comparison CSV", button_style="success")
export_emi_btn = widgets.Button(description="Export EMI CSV", button_style="success")

# Output areas
main_output = widgets.Output()
spec_out = widgets.Output()        # dedicated output for immediate spec display
summary_out = widgets.Output()

# Globals for export
_last_comparison_df = None
_last_emi_df = None

# Populate compare selector (keeps full list for multi-select)
def populate_compare_selector():
    all_pairs = []
    for b in sorted(BIKES.keys()):
        for m in sorted(BIKES[b].keys()):
            all_pairs.append((f"{b} {m}", (b, m)))
    compare_selector.options = all_pairs

populate_compare_selector()

# Show specs immediately when both brand and model are selected
def show_selected_specs(brand, model):
    with spec_out:
        spec_out.clear_output()
        if not brand or not model:
            print("Select a brand and model to view specifications.")
            return
        specs_dict = BIKES[brand][model].copy()
        display_dict = specs_dict.copy()
        if "Ex-Showroom Price" in display_dict:
            display_dict["Ex-Showroom Price"] = format_currency(display_dict["Ex-Showroom Price"])
        df = pd.DataFrame.from_dict(display_dict, orient="index", columns=[f"{brand} {model}"])
        df.index.name = "Specification"
        display_df = df.astype(object)
        display(HTML(f"<h4>Specifications: {brand} {model}</h4>"))
        display(display_df)

# Update model list when brand changes and set down payment max to price
def on_brand_change(change):
    brand = change['new']
    if not brand:
        # reset model dropdown to placeholder
        model_dropdown.options = [("Select Model", None)]
        model_dropdown.value = None
        down_payment_input.max = 1e9
        with spec_out:
            spec_out.clear_output()
            print("Select a brand and model to view specifications.")
        return
    models = sorted(BIKES[brand].keys())
    model_dropdown.options = [("Select Model", None)] + [(m, m) for m in models]
    model_dropdown.value = None
    down_payment_input.max = 1e9
    with spec_out:
        spec_out.clear_output()
        print("Select a model to view specifications.")

brand_dropdown.observe(on_brand_change, names='value')

# Update down payment max and show specs when model changes
def on_model_change(change):
    brand = brand_dropdown.value
    model = change['new']
    if not brand or not model:
        with spec_out:
            spec_out.clear_output()
            print("Select a brand and model to view specifications.")
        down_payment_input.max = 1e9
        return
    price = float(BIKES[brand][model]["Ex-Showroom Price"])
    down_payment_input.max = price
    show_selected_specs(brand, model)

model_dropdown.observe(on_model_change, names='value')

# Handlers for compare and EMI (main_output used for larger tables & exports)
def on_compare_clicked(b):
    global _last_comparison_df
    with main_output:
        main_output.clear_output()
        selected = list(compare_selector.value)
        if len(selected) < 2:
            print("Select at least two bikes to compare.")
            return
        df = build_comparison_df(selected)
        df_display = df.copy().astype(object)
        if "Ex-Showroom Price" in df_display.index:
            df_display.loc["Ex-Showroom Price"] = df.loc["Ex-Showroom Price"].apply(format_currency)
        _last_comparison_df = df_display
        display(HTML("<h3>Bike Comparison</h3>"))
        display(df_display)
        # Quick summary
        try:
            prices = df.loc["Ex-Showroom Price"].astype(float)
            cheapest = prices.idxmin()
            power = df.loc["Max Power bhp"].astype(float)
            most_powerful = power.idxmax()
            mileage = df.loc["Mileage kmpl"].astype(float)
            best_mileage = mileage.idxmax()
            with summary_out:
                summary_out.clear_output()
                display(HTML("<h4>Quick Summary</h4>"))
                print(f"Cheapest: {cheapest}")
                print(f"Most Powerful: {most_powerful}")
                print(f"Best Mileage: {best_mileage}")
        except Exception:
            pass

def on_emi_clicked(b):
    global _last_emi_df
    with main_output:
        main_output.clear_output()
        brand = brand_dropdown.value
        model = model_dropdown.value
        if not brand or not model:
            print("Select a brand and model first.")
            return
        dp = float(down_payment_input.value)
        rate = float(interest_input.value)
        emi_df = emi_breakdown_for_model(brand, model, down_payment=dp, annual_rate=rate)
        _last_emi_df = emi_df
        display(HTML(f"<h3>EMI Breakdown for {brand} {model}</h3>"))
        display(emi_df)

def export_df_to_csv(df, filename):
    buffer = io.StringIO()
    df.to_csv(buffer)
    buffer.seek(0)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(buffer.getvalue())
    return filename

def on_export_compare(b):
    with main_output:
        if _last_comparison_df is None:
            print("No comparison to export.")
            return
        fname = export_df_to_csv(_last_comparison_df, "bike_comparison.csv")
        display(FileLink(fname))

def on_export_emi(b):
    with main_output:
        if _last_emi_df is None:
            print("No EMI table to export.")
            return
        fname = export_df_to_csv(_last_emi_df, "emi_breakdown.csv")
        display(FileLink(fname))

# Wire handlers
compare_button.on_click(on_compare_clicked)
emi_button.on_click(on_emi_clicked)
export_compare_btn.on_click(on_export_compare)
export_emi_btn.on_click(on_export_emi)

# No default selection: user must pick brand and model
brand_dropdown.value = None
model_dropdown.value = None

# Layout: include spec_out so specs appear immediately after selection
controls = widgets.VBox([
    widgets.HBox([brand_dropdown, model_dropdown]),
    widgets.HBox([down_payment_input, interest_input, emi_button]),
    widgets.HBox([compare_selector, compare_button]),
    widgets.HBox([export_compare_btn, export_emi_btn])
])

display(controls, spec_out, main_output, summary_out)

Output()

Output()

Output()